# Capitalisations à prendre
## Méga-capitalisations (groupe contrôle, 12)
- AAPL
- MSFT


## Intermédiaires

In [34]:
import yfinance as yf
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime
import requests
from time import sleep
import finnhub 
from dotenv import load_dotenv
import os
from fmp_python.fmp import FMP
from requests import JSONDecodeError

In [2]:
# values

db_path = '../data/prices.db'
start="2025-01-01"
end="2025-12-31"

In [3]:
k = yf.download('AAPL',start=start, end=end, auto_adjust=False).dropna(axis=1, how='all').stack().reset_index()


k = k.rename(columns={
    'level_0': 'date', 'Ticker': 'ticker', 'Open': 'open', 'Close': 'close', 'High': 'high', 'Low': 'low', 'Adj Close': 'adj_close', 'Volume': 'volume'
})

k

[*********************100%***********************]  1 of 1 completed


Price,date,ticker,adj_close,close,high,low,open,volume
0,2025-01-02,AAPL,242.301941,243.850006,249.100006,241.820007,248.929993,55740700
1,2025-01-03,AAPL,241.815033,243.360001,244.179993,241.889999,243.360001,40244100
2,2025-01-06,AAPL,243.444626,245.000000,247.330002,243.199997,244.309998,45045600
3,2025-01-07,AAPL,240.672333,242.210007,245.550003,241.350006,242.979996,40856000
4,2025-01-08,AAPL,241.159210,242.699997,243.710007,240.050003,241.919998,37628900
...,...,...,...,...,...,...,...,...
244,2025-12-23,AAPL,271.854919,272.359985,272.500000,269.559998,270.839996,29642000
245,2025-12-24,AAPL,273.302216,273.809998,275.429993,272.200012,272.339996,17910600
246,2025-12-26,AAPL,272.892975,273.399994,275.369995,272.859985,274.160004,21521800
247,2025-12-29,AAPL,273.252350,273.760010,274.359985,272.350006,272.690002,23715200


In [20]:
# CORRECTION


def init_db(db_path):
    with duckdb.connect(db_path) as con:
        # create table
        con.execute("""
                    CREATE TABLE IF NOT EXISTS prices ( 
                    date DATE NOT NULL, 
                    ticker VARCHAR NOT NULL, 
                    adj_close DOUBLE,
                    close DOUBLE, 
                    high DOUBLE, 
                    low DOUBLE,  
                    open DOUBLE, 
                    volume BIGINT,
                    PRIMARY KEY (date, ticker)
                    );
                    """
        )


def fetch_prices(tickers, start, end):
    frames = []

    for ticker in tickers:
        try:
            rows = yf.download(ticker,start=start, end=end, auto_adjust=False).reset_index()
            if rows.empty:
                print(f"[skip] {ticker}: no data")
                continue
            
            # Map columns correctly
            if isinstance(rows.columns, pd.MultiIndex):
                rows.columns = rows.columns.get_level_values(0)
            rows['ticker'] = ticker
            rows = rows.rename(columns={
                'index': 'date', 'Ticker': 'ticker', 'Open': 'open', 'Close': 'close', 'High': 'high', 'Low': 'low', 'Adj Close': 'adj_close', 'Volume': 'volume'
            })
            # Remove NA values if any
            prev_lines = rows.shape[0]
            rows = rows.dropna(subset=['close', 'adj_close'])
            if rows.shape[0] < prev_lines:
                print(f'Removed {rows.shape[0] - prev_lines} lines due to NA values in close/adj_close')

            frames.append(rows)
        except Exception as e:
            print(f'An error occurred for ticker {ticker}: {e}. \n Continuing...')
            continue
        finally:
            sleep(0.3)
    if not frames:
        print("No data fetched for any ticker.")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def store_prices(df, db_path):
    with duckdb.connect(db_path) as con:
        con.register('my_df', df)
        con.execute('INSERT OR REPLACE INTO prices (date, ticker, adj_close, close, high, low, open, volume) SELECT date, ticker, adj_close, close, high, low, open, volume FROM my_df')
    
def query_prices(ticker, db_path) -> pd.DataFrame:
    with duckdb.connect(db_path) as con:
        query = 'SELECT * FROM prices WHERE ticker = ?'
        output = con.execute(query, [ticker]).fetchdf()
        if output.empty:
            print(f'Specified ticker {ticker} returned empty')
    return output

In [21]:
# testing

tickers = ['AAPL', 'hOIUHFNOIU', 'MSFT', 'SOFI', 'LCID', 'GOOG', 'TTE']

init_db(db_path)
df = fetch_prices(tickers, start, end)
store_prices(df, db_path)
query_prices('AAPL', db_path)


[*********************100%***********************]  1 of 1 completed
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HOIUHFNOIU"}}}
$HOIUHFNOIU: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['HOIUHFNOIU']: possibly delisted; no timezone found


[skip] hOIUHFNOIU: no data


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,date,ticker,adj_close,close,high,low,open,volume
0,2025-01-27,AAPL,228.400726,229.860001,232.149994,223.979996,224.020004,94863400
1,2025-01-28,AAPL,236.747406,238.259995,240.190002,230.809998,230.850006,75707600
2,2025-01-31,AAPL,234.501740,236.000000,247.190002,233.440002,247.190002,100959800
3,2025-02-11,AAPL,231.397354,232.619995,235.229996,228.130005,228.199997,53718400
4,2025-02-12,AAPL,235.625000,236.869995,236.960007,230.679993,231.199997,45243300
...,...,...,...,...,...,...,...,...
244,2025-11-10,AAPL,268.930359,269.429993,273.730011,267.459991,268.959991,41312400
245,2025-11-11,AAPL,274.739563,275.250000,275.910004,269.799988,269.809998,46208300
246,2025-11-14,AAPL,271.904816,272.410004,275.959991,269.600006,271.049988,47431300
247,2025-12-11,AAPL,277.514404,278.029999,279.589996,273.809998,279.100006,33248000


In [ ]:
raw = yf.download('AAPL', start='2024-01-01', end='2024-02-01', auto_adjust=False)
print("Type colonnes:", type(raw.columns))
print("Colonnes:", raw.columns.tolist())
print("Après reset_index:", raw.reset_index().columns.tolist())

[*********************100%***********************]  1 of 1 completed

Type colonnes: <class 'pandas.MultiIndex'>
Colonnes: [('Adj Close', 'AAPL'), ('Close', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Volume', 'AAPL')]
Après reset_index: [('index', ''), ('Adj Close', 'AAPL'), ('Close', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Volume', 'AAPL')]


In [23]:
raw = yf.download('AAPL', start='2024-01-01', end='2024-02-01', auto_adjust=False)
print(raw.columns.get_level_values(0))
print("Type colonnes:", type(raw.columns))
print("Colonnes:", raw.columns.tolist())
print("Après reset_index:", raw.reset_index().columns.tolist())

[*********************100%***********************]  1 of 1 completed

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')
Type colonnes: <class 'pandas.MultiIndex'>
Colonnes: [('Adj Close', 'AAPL'), ('Close', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Volume', 'AAPL')]
Après reset_index: [('index', ''), ('Adj Close', 'AAPL'), ('Close', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Volume', 'AAPL')]


In [24]:
init_db(db_path)
df = fetch_prices(['AAPL', 'MSFT', 'SOFI', 'LCID', 'GOOG', 'TTE'], start, end)
store_prices(df, db_path)

aapl = query_prices('AAPL', db_path)
lcid = query_prices('LCID', db_path)
print(aapl[['date','close']].head())
print(lcid[['date','close']].head())
# AAPL doit être autour de 180-250$, LCID autour de 2-4$. S'ils sont identiques, bug.

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


        date       close
0 2025-01-27  229.860001
1 2025-01-28  238.259995
2 2025-01-31  236.000000
3 2025-02-11  232.619995
4 2025-02-12  236.869995
        date      close
0 2025-01-27  27.500000
1 2025-01-28  27.700001
2 2025-01-31  27.600000
3 2025-02-11  26.900000
4 2025-02-12  28.700001


Loop over tickers, pull each from yfinance, normalize into your chosen long format with consistent column names. Handle the cases where a ticker returns nothing or errors out (one bad ticker shouldn't kill the whole run). Be polite with a small sleep between requests. Heads-up on a real gotcha: recent yfinance versions return a MultiIndex column frame even for a single ticker — you'll need to flatten it, and discovering that yourself by inspecting df.columns is a good exercise.

## The spec — implement these four functions:

init_db(db_path) — connect to DuckDB, create a prices table if it doesn't exist with columns for ticker, date, the OHLC prices, adjusted close, and volume. Decide the primary key. Think about types (what's the right type for volume vs. price?).

fetch_prices(tickers, start, end) -> DataFrame — loop over tickers, pull each from yfinance, normalize into your chosen long format with consistent column names. Handle the cases where a ticker returns nothing or errors out (one bad ticker shouldn't kill the whole run). Be polite with a small sleep between requests. Heads-up on a real gotcha: recent yfinance versions return a MultiIndex column frame even for a single ticker — you'll need to flatten it, and discovering that yourself by inspecting df.columns is a good exercise.

store_prices(df, db_path) — write the dataframe into DuckDB idempotently. Look up how DuckDB lets you query a registered pandas dataframe directly; it's elegant and you'll use the pattern repeatedly.

query_prices(ticker, db_path) -> DataFrame — read one ticker's history back, date-ordered. Use a parameterized query, not string formatting — I want you to build the safe-SQL habit from day one even though this is local.
How to validate your work: write a small script that fetches maybe five tickers (mix a mega-cap like MSFT with something small and volatile like SOFI), stores them, reads one back, and prints the date range and row count. If you can round-trip real data, the layer works.

In [1]:
# init_db

# create a persistent DB

# create prices table : ticker, date, OHLC, adjusted close, volume. Primary key should be (ticket, date) ? or add an ID ?
# volume : int ; price : float


# 6. Ce que tu vas construire cette semaine
L'objectif : une table earnings dans DuckDB, avec une ligne par (ticker, date d'annonce), contenant le BPA réalisé, le BPA estimé, la surprise, et le SUE calculé.
Décisions de conception à trancher avant de coder :
D'abord, le schéma de la table earnings. Réfléchis aux colonnes : ticker, date d'annonce, date de fin de période fiscale (garde-la, utile pour vérifier le décalage), BPA réalisé, BPA estimé, surprise brute, SUE. Pense à la primary key — qu'est-ce qui identifie une ligne de façon unique ?
Ensuite, la source. Je te suggère de commencer avec Finnhub (free tier, inscription gratuite, clé API). Tu fais un appel par ticker sur l'endpoint earnings, tu récupères l'historique trimestriel. Mets ta clé API dans un .env (jamais en dur dans le code — tu as déjà .env dans ton .gitignore, donc utilise python-dotenv pour la charger).
Puis, le calcul du SUE. Pour chaque (ticker, trimestre) : surprise = actual − estimate ; puis SUE = surprise / écart-type des 8 dernières surprises de ce ticker. Attention : le SUE d'un trimestre ne doit utiliser que les surprises passées pour l'écart-type (sinon, encore une fois, look-ahead). C'est un calcul en fenêtre glissante expanding ou rolling sur les données triées par date.
Enfin, la gestion du fallback quand un ticker a moins de 8 trimestres d'historique : soit tu utilises l'écart-type sur ce qui est disponible (avec un minimum, genre 4 trimestres), soit tu bascules sur la division par le prix, soit tu marques le SUE comme non calculable. Choisis et assume.
Le spec des fonctions (à toi de les écrire) :
fetch_earnings(tickers, api_key) → récupère l'historique earnings depuis Finnhub pour chaque ticker, retourne un DataFrame long avec ticker, date d'annonce, période, actual, estimate. Gère le rate-limit (sleep) et les erreurs par ticker comme tu l'as appris.
compute_sue(df) → prend le DataFrame d'earnings, trie par ticker et date, calcule la surprise puis le SUE en fenêtre glissante par ticker (attention au look-ahead dans le calcul de σ). Retourne le DataFrame enrichi.
init_earnings_db(db_path) et store_earnings(df, db_path) → même logique que pour les prix, tu as déjà le pattern.
Avant que tu te lances, trois questions pour vérifier que la théorie est passée :

Pourquoi est-ce qu'on standardise la surprise au lieu d'utiliser la surprise brute en dollars ?
Si une entreprise clôt son trimestre le 30 juin et publie ses résultats le 28 juillet, quelle date utilises-tu comme référence pour ton signal, et pourquoi l'autre serait une erreur ?
Dans le calcul de l'écart-type pour standardiser le SUE du trimestre actuel, pourquoi est-ce que je ne peux pas inclure la surprise du trimestre actuel lui-même dans le calcul de σ ?

Réponds à ces trois-là (briève­ment), et si c'est bon, tu attaques le code. On reste en français.
                        
                        La théorie est solide. Vas-y, code la couche earnings :

init_earnings_db + store_earnings (tu as le pattern des prix).
fetch_earnings(tickers, api_key) via Finnhub, clé dans .env chargée avec python-dotenv, rate-limit géré, erreurs par ticker.
compute_sue(df) : tri par ticker/date, surprise = actual − estimate, σ en rolling 8 trimestres décalé d'un cran (pour exclure le trimestre courant — réfléchis à comment .shift() t'aide ici), SUE = surprise / σ, NA si historique insuffisant.

In [ ]:
def init_db(db_path):
    with duckdb.connect(db_path) as con:
        # create table
        con.execute("""
                    CREATE TABLE IF NOT EXISTS earnings ( 
                    announcement DATE NOT NULL,
                    ticker VARCHAR NOT NULL, 
                    realized_eps DOUBLE,
                    expected_eps DOUBLE,
                    surprise DOUBLE,
                    sue DOUBLE,
                    expected_revenue DOUBLE,
                    realized_revenue DOUBLE,
                    is_future BOOLEAN,
                    last_updated DATE,
                    PRIMARY KEY (announcement, ticker)
                    );
                    """
        )

def store_earnings(df, db_path):
    with duckdb.connect(db_path) as con:
        con.register('my_df', df)
        con.execute('INSERT OR REPLACE INTO earnings (announcement, ticker, realized_eps, expected_eps, surprise, sue, expected_revenue, realized_revenue, is_future, last_updated) SELECT announcement, ticker, realized_eps, expected_eps, surprise, sue, expected_revenue, realized_revenue, is_future, last_updated FROM my_df')

def fetch_earnings(tickers, api_key):
    earnings_df = []
    endpoint = 'https://financialmodelingprep.com/stable/earnings'
    # (announcement, ticker, realized_eps, expected_eps, surprise, sue, expected_revenue, realized_revenue, is_future, last_updated)
    for ticker in tickers:
        print(f'Fetching for {ticker}...')
        url = f'{endpoint}?symbol={ticker}&apikey={api_key}'
        try:
            r = requests.get(url)
            if r.status_code == 200:
                if r.json() == []:
                    print(f'error HTTPS 200 BUT json is empty! For ticker {ticker}')
                    continue
                df = pd.DataFrame(r.json())
                # process df symbol	date	epsActual	epsEstimated	revenueActual	revenueEstimated	lastUpdated
                df['is_future'] = df['epsActual'].isna()
                df = df.rename(columns={'symbol': 'ticker',
                        'date': 'announcement',
                        'epsActual': 'realized_eps',
                        'epsEstimated': 'expected_eps',
                        'revenueEstimated': 'expected_revenue',
                            'revenueActual': 'realized_revenue',
                            'lastUpdated': 'last_updated'
                        })
                # calculate sue and surprise
                df['surprise'] = df.realized_eps - df.expected_eps
                earnings_df.append(df)
            else:
                print(f'Error on HTTP request {r.status_code}')
            
        except Exception as e:
            print(f'Error for ticker {ticker}: {e}')
        sleep(1)
    return pd.concat(earnings_df).sort_values(by='announcement', ascending=True) # sorted ancient to newest


def compute_sue(df, window=8, min_periods=4):
    df = df.sort_values(['ticker', 'announcement']).copy()
    def _rolling_sigma(s):
        return s.shift(1).rolling(window=window, min_periods=min_periods).std(ddof=0)
    sigma = df.groupby(by='ticker').surprise.transform(_rolling_sigma)
    df['sue'] = df['surprise'] / sigma
    return df


In [316]:
df = fetch_earnings(['TSLA'], os.getenv('FMP_API_KEY'))
df = compute_sue(df)


Fetching for TSLA...


In [317]:
print(df[['announcement', 'surprise', 'sue']].iloc[-32:-20])

   announcement  surprise       sue
31   2018-10-24   0.15986  7.105094
30   2019-01-30  -0.01000 -0.177652
29   2019-04-24  -0.11000 -1.960399
28   2019-07-24  -0.04772 -0.673647
27   2019-10-23   0.10133  1.416247
26   2020-01-29   0.03000  0.381740
25   2020-04-29   0.06563  0.832740
24   2020-07-22   0.13916  1.732780
23   2020-10-21   0.07000  0.798998
22   2021-01-27  -0.03000 -0.390925
21   2021-04-26   0.05000  0.638515
20   2021-07-26   0.15000  2.557504


In [314]:
df[df.ticker=='AAPL'][['announcement','realized_eps','expected_eps','surprise']].iloc[-32:-20]

,announcement,realized_eps,expected_eps,surprise
31,2018-11-01,0.73,0.70,0.03
30,2019-01-29,1.04,1.04,0.00
29,2019-04-30,0.62,0.59,0.03
28,2019-07-30,0.55,0.52,0.03
27,2019-10-30,0.76,0.71,0.05
26,2020-01-28,1.25,1.13,0.12
25,2020-04-30,0.64,0.52,0.12
24,2020-07-30,0.64,0.51,0.13
23,2020-10-29,0.73,0.69,0.04
22,2021-01-27,1.68,1.41,0.27


In [315]:
aapl = df[df.ticker=='AAPL'].sort_values('announcement').reset_index(drop=True)
i = aapl.index[aapl.announcement=='2019-04-30'][0]
fenetre = aapl['surprise'].iloc[max(0,i-8):i]
print("fenêtre:", fenetre.values)
print("points non-NaN:", fenetre.notna().sum())
print("sigma pop:", fenetre.std(ddof=0))
print("sue recalc:", aapl['surprise'].iloc[i] / fenetre.std(ddof=0))

fenêtre: [0.02 0.03 0.05 0.02 0.01 0.04 0.03 0.  ]
points non-NaN: 8
sigma pop: 0.014999999999999998
sue recalc: 2.000000000000002


In [287]:
k = fetch_earnings(['AAPL', 'TSLA'], os.getenv('FMP_API_KEY'))

Fetching for AAPL...
Fetching for TSLA...


In [288]:
k = compute_sue(k)
store_earnings(k, '../data/earnings.db')
k

,ticker,announcement,realized_eps,expected_eps,realized_revenue,expected_revenue,last_updated,is_future,surprise,sue
163,AAPL,1985-09-30,0.00161,NaN,4.097000e+08,NaN,2026-03-23,False,NaN,NaN
162,AAPL,1985-12-31,0.00411,NaN,5.339000e+08,NaN,2026-03-23,False,NaN,NaN
161,AAPL,1986-03-31,0.00227,NaN,4.089000e+08,NaN,2026-03-23,False,NaN,NaN
160,AAPL,1986-06-30,0.00227,NaN,4.483000e+08,NaN,2026-03-23,False,NaN,NaN
159,AAPL,1986-09-30,0.00235,NaN,5.108000e+08,NaN,2026-03-23,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2,AAPL,2026-01-29,2.85000,2.6700,1.437560e+11,1.383910e+11,2026-06-04,False,0.1800,4.502199
1,TSLA,2026-04-22,0.41000,0.3539,2.238700e+10,2.210078e+10,2026-06-13,False,0.0561,0.675040
1,AAPL,2026-04-30,2.01000,1.9500,1.111840e+11,1.094576e+11,2026-06-13,False,0.0600,1.077903
0,TSLA,2026-07-22,NaN,0.4500,NaN,2.430876e+10,2026-06-13,True,NaN,NaN


In [289]:
with duckdb.connect('../data/earnings.db') as con:
    n = con.execute('SELECT * from earnings').fetchdf()
n

,announcement,ticker,realized_eps,expected_eps,surprise,sue,expected_revenue,realized_revenue,is_future,last_updated
0,1988-06-30,AAPL,0.00624,NaN,NaN,NaN,NaN,9.931000e+08,False,2026-03-23
1,1988-12-31,AAPL,0.00984,NaN,NaN,NaN,NaN,1.405100e+09,False,2026-03-23
2,1992-06-30,AAPL,0.00964,NaN,NaN,NaN,NaN,1.740200e+09,False,2026-03-23
3,1993-01-16,AAPL,0.01190,NaN,NaN,NaN,NaN,2.000292e+09,False,2026-06-04
4,1994-01-21,AAPL,0.00310,NaN,NaN,NaN,NaN,2.469000e+09,False,2026-06-04
...,...,...,...,...,...,...,...,...,...,...
233,2023-07-19,TSLA,0.91000,0.820,0.090,0.975112,2.447710e+10,2.492700e+10,False,2025-04-25
234,2024-02-01,AAPL,2.18000,2.100,0.080,1.174955,1.179866e+11,1.195750e+11,False,2026-06-04
235,2025-01-29,TSLA,0.73000,0.774,-0.044,-0.556630,2.725892e+10,2.570700e+10,False,2025-04-29
236,2025-01-30,AAPL,2.40000,2.360,0.040,0.970589,1.242574e+11,1.243000e+11,False,2026-06-04


In [4]:
# comment finnhub

finnhub_client = finnhub.Client(api_key=os.getenv('FINNHUB_API_KEY'))

In [ ]:
k = finnhub_client.company_earnings('SPCX', limit=None)

In [11]:
fmp = FMP(api_key=os.getenv('FMP_API_KEY'))

In [219]:
r2

[{'symbol': 'LCID',
  'date': '2026-08-04',
  'epsActual': None,
  'epsEstimated': -2.40034,
  'revenueActual': None,
  'revenueEstimated': 441916700,
  'lastUpdated': '2026-06-13'},
 {'symbol': 'LCID',
  'date': '2026-05-05',
  'epsActual': -3.46,
  'epsEstimated': -2.72,
  'revenueActual': 282465000,
  'revenueEstimated': 358455600,
  'lastUpdated': '2026-06-13'},
 {'symbol': 'LCID',
  'date': '2026-02-24',
  'epsActual': -3.62,
  'epsEstimated': -2.49,
  'revenueActual': 522730000,
  'revenueEstimated': 459530700,
  'lastUpdated': '2026-05-24'},
 {'symbol': 'LCID',
  'date': '2025-11-05',
  'epsActual': -3.31,
  'epsEstimated': -2.32,
  'revenueActual': 336580000,
  'revenueEstimated': 473078750,
  'lastUpdated': '2026-02-05'},
 {'symbol': 'LCID',
  'date': '2025-08-05',
  'epsActual': -2.8,
  'epsEstimated': -2.18,
  'revenueActual': 259432000,
  'revenueEstimated': 259354682,
  'lastUpdated': '2025-11-05'},
 {'symbol': 'LCID',
  'date': '2025-05-06',
  'epsActual': -0.24,
  'epsEs

In [220]:

key = os.getenv('FMP_API_KEY')
# Surprises historiques
url = f"https://financialmodelingprep.com/stable/earnings?symbol=AAPL&apikey={key}"
r2 = requests.get(url).json()
print(len(r2), "trimestres disponibles")
print(r2[:3])

164 trimestres disponibles
[{'symbol': 'AAPL', 'date': '2026-07-30', 'epsActual': None, 'epsEstimated': 1.86, 'revenueActual': None, 'revenueEstimated': 108400600000, 'lastUpdated': '2026-06-13'}, {'symbol': 'AAPL', 'date': '2026-04-30', 'epsActual': 2.01, 'epsEstimated': 1.95, 'revenueActual': 111184000000, 'revenueEstimated': 109457600000, 'lastUpdated': '2026-06-13'}, {'symbol': 'AAPL', 'date': '2026-01-29', 'epsActual': 2.85, 'epsEstimated': 2.67, 'revenueActual': 143756000000, 'revenueEstimated': 138391000000, 'lastUpdated': '2026-06-04'}]


In [225]:
pd.DataFrame(r2).tail(40)

,symbol,date,epsActual,epsEstimated,revenueActual,revenueEstimated,lastUpdated
124,AAPL,1995-07-20,0.00750,0.0075,2.575000e+09,2.575000e+09,2026-06-04
125,AAPL,1995-04-20,0.00530,0.0100,2.652000e+09,2.575000e+09,2026-06-04
126,AAPL,1995-01-19,0.01380,NaN,2.832000e+09,NaN,2026-06-04
127,AAPL,1994-12-13,0.00850,NaN,2.493286e+09,NaN,2026-06-04
128,AAPL,1994-10-17,0.01040,NaN,2.149908e+09,NaN,2026-06-04
129,AAPL,1994-04-21,0.00130,NaN,2.077000e+09,NaN,2026-06-04
130,AAPL,1994-01-21,0.00310,NaN,2.469000e+09,NaN,2026-06-04
131,AAPL,1993-10-15,0.00021,NaN,2.140800e+09,NaN,2026-06-04
132,AAPL,1993-07-16,-0.01450,NaN,1.861979e+09,NaN,2026-06-04
133,AAPL,1993-04-20,0.00820,NaN,1.973900e+09,NaN,2026-06-04


In [194]:
kf = pd.DataFrame(r2)

kf['sue'] = np.nan
kf

,symbol,date,epsActual,epsEstimated,revenueActual,revenueEstimated,lastUpdated,sue
0,LCID,2026-08-04,NaN,-2.40034,NaN,441916700.0,2026-06-13,NaN
1,LCID,2026-05-05,-3.46000,-2.72000,282465000.0,358455600.0,2026-06-13,NaN
2,LCID,2026-02-24,-3.62000,-2.49000,522730000.0,459530700.0,2026-05-24,NaN
3,LCID,2025-11-05,-3.31000,-2.32000,336580000.0,473078750.0,2026-02-05,NaN
4,LCID,2025-08-05,-2.80000,-2.18000,259432000.0,259354682.0,2025-11-05,NaN
5,LCID,2025-05-06,-0.24000,-0.23000,235048000.0,246164380.0,2025-08-06,NaN
6,LCID,2025-02-25,-0.22000,-0.26000,234473000.0,211766140.0,2025-05-25,NaN
7,LCID,2024-11-07,-0.28000,-0.30000,200038000.0,197970000.0,2024-11-08,NaN
8,LCID,2024-08-05,-0.29000,-0.27000,200581000.0,192060000.0,2025-04-25,NaN
9,LCID,2024-05-06,-0.30000,-0.25000,172740000.0,155990000.0,2025-04-25,NaN


In [196]:
kf.epsActual.shift(-1)[::-1].rolling(window=8).std(ddof=0)[::-1]

0      1.535600
1      1.456389
2      1.212930
3      0.836641
4      0.049687
5      0.064505
6      0.059987
7      0.059674
8      0.056111
9      0.115515
10     0.115731
11     0.116344
12     0.129204
13     4.048253
14    10.119369
15     9.930750
16     9.938244
17     9.740299
18          NaN
19          NaN
20          NaN
21          NaN
22          NaN
23          NaN
24          NaN
25          NaN
Name: epsActual, dtype: float64

In [44]:
df = pd.DataFrame(r)
df

,symbol,date,epsActual,epsEstimated,revenueActual,revenueEstimated,lastUpdated
0,TSLA,2026-07-22,NaN,0.4500,NaN,2.430876e+10,2026-06-13
1,TSLA,2026-04-22,0.41000,0.3539,2.238700e+10,2.210078e+10,2026-06-13
2,TSLA,2026-01-28,0.50000,0.4548,2.490100e+10,2.477644e+10,2026-04-28
3,TSLA,2025-10-22,0.50000,0.5580,2.809500e+10,2.654037e+10,2026-01-22
4,TSLA,2025-07-23,0.40000,0.3972,2.249600e+10,2.227968e+10,2025-10-23
...,...,...,...,...,...,...,...
69,TSLA,2009-03-31,-0.00763,NaN,2.088600e+07,NaN,2025-04-25
70,TSLA,2008-12-31,NaN,NaN,1.416200e+07,NaN,2025-04-25
71,TSLA,2008-03-31,-0.01012,NaN,3.685500e+06,NaN,2025-04-25
72,TSLA,2008-01-31,-0.01012,NaN,3.685500e+06,NaN,2025-04-25


In [59]:
init_db('../data/earnings.db')
with duckdb.connect('../data/earnings.db') as con:
    df = con.execute('SELECT * FROM earnings').fetchdf()

In [97]:
from copy import deepcopy

In [99]:
m = deepcopy(r)
for rrrow in m:
    rrrow['symbol'] = 'AAPL'
